In [ ]:
# --- CELLULE 1 : Installation des bibliothèques et montage du Drive ---

# 1. On installe les outils d'Intelligence Artificielle d'HuggingFace
# (jiwer et evaluate serviront à calculer ton taux d'erreur - CER)
!pip install -q transformers datasets accelerate evaluate jiwer

# 2. On connecte ta machine Colab à ton Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Étape 1 terminée : Outils installés et Drive connecté !")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Étape 1 terminée : Outils installés et Drive connecté !


In [ ]:
# --- CELLULE 2 : Préparation des données (La classe PyTorch) ---

import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image
from transformers import TrOCRProcessor

# 1. Tes chemins de fichiers (Vérifie que c'est bien le nom de ton dossier Drive)
ROOT_DIR = "/content/drive/MyDrive/OCR_Prescription"
MODEL_NAME = "microsoft/trocr-base-handwritten"

# 2. Définition du "Traducteur" (Dataset PyTorch)
class PrescriptionDataset(Dataset):
    def __init__(self, csv_file, image_folder, processor, max_target_length=128):
        self.df = pd.read_csv(csv_file)
        self.df['text'] = self.df['text'].astype(str) # Sécurité : forcer le format texte
        self.image_folder = image_folder
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # On récupère le nom et le texte
        file_name = self.df.iloc[idx]['file_name']
        text = self.df.iloc[idx]['text']

        # On charge l'image
        img_path = os.path.join(self.image_folder, file_name)
        image = Image.open(img_path).convert("RGB")

        # Le Processor transforme l'image pour le réseau de neurones
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        # Le Tokenizer transforme le texte en numéros
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True
        ).input_ids

        # Astuce technique : on remplace le vide par -100 pour que l'IA ne l'apprenne pas
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}

# 3. Chargement du "Cerveau linguistique" (Processor)
print("Téléchargement du processeur de Microsoft...")
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)

# 4. Création de tes deux datasets
print("Création du pont avec tes données Drive...")
train_dataset = PrescriptionDataset(
    csv_file=os.path.join(ROOT_DIR, "train_trocr.csv"),
    image_folder=os.path.join(ROOT_DIR, "train/images"),
    processor=processor
)

val_dataset = PrescriptionDataset(
    csv_file=os.path.join(ROOT_DIR, "val_trocr.csv"),
    image_folder=os.path.join(ROOT_DIR, "val/images"),
    processor=processor
)

# 5. TEST DE SÉCURITÉ (On vérifie que la première image charge bien)
print("\n" + "="*40)
print(f"✅ Succès ! Tu as {len(train_dataset)} images pour l'entraînement.")
print(f"✅ Succès ! Tu as {len(val_dataset)} images pour la validation.")
print("="*40)

Téléchargement du processeur de Microsoft...
Création du pont avec tes données Drive...

✅ Succès ! Tu as 3120 images pour l'entraînement.
✅ Succès ! Tu as 780 images pour la validation.


In [ ]:
# --- CELLULE 3 : Configuration et Lancement de l'Entraînement ---

import evaluate
import numpy as np
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator
from transformers import VisionEncoderDecoderModel

# 1. Chargement du modèle de base (le "cerveau" vierge)
print("Chargement de l'architecture TrOCR...")
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# Paramètres spéciaux pour que le modèle comprenne où commencent et finissent les phrases
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# 2. Définition de la métrique d'évaluation (CER)
print("Chargement de la métrique d'évaluation (CER)...")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    # Remplacer les -100 (les vides) par le token de padding normal pour le décodage
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id

    # Traduire les numéros prédits en texte lisible
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    # Calcul du taux d'erreur
    cer = cer_metric.compute(predictions=pred_str, references=labels_str)
    return {"cer": cer}

# 3. Configuration des Hyperparamètres
print("Configuration du moteur d'entraînement...")
training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(ROOT_DIR, "trocr_checkpoints"), # Dossier des sauvegardes intermédiaires
    predict_with_generate=True,
    eval_strategy="steps",
    eval_steps=200,                  # Évalue toutes les 200 étapes
    save_steps=200,
    logging_steps=50,                # Affiche l'avancement toutes les 50 étapes
    per_device_train_batch_size=8,   # ⚠️ Si Colab crash avec "CUDA Out of Memory", passe ceci à 4
    per_device_eval_batch_size=8,    # ⚠️ Et ceci à 4 également
    fp16=True,                       # Utilisation de la précision mixte (accélération matérielle GPU)
    learning_rate=4e-5,              # Taux d'apprentissage standard pour du fine-tuning
    num_train_epochs=3,              # Le modèle va étudier tes 3120 images 3 fois au total
    save_total_limit=2,              # Ne garde que les 2 meilleures sauvegardes pour économiser l'espace Drive
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,         # Pour le CER, plus le chiffre est bas, meilleur est le modèle
    report_to="none"                 # Désactiver les logs externes pour simplifier
)

# 4. Lancement du Trainer
trainer = Seq2SeqTrainer(
    model=model,
    processing_class=processor,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator,
)

print("\n🚀 DÉMARRAGE DE L'ENTRAÎNEMENT !")
print("Une barre de progression va apparaître. Surveille la colonne 'cer' !")
trainer.train()

# 5. Sauvegarde finale
print("\n💾 Sauvegarde du modèle final sur Google Drive...")
MODELE_FINAL_PATH = os.path.join(ROOT_DIR, "modele_trocr_medical_final")
trainer.save_model(MODELE_FINAL_PATH)
processor.save_pretrained(MODELE_FINAL_PATH)
print(f"✅ Entraînement terminé ! Modèle sauvegardé dans : {MODELE_FINAL_PATH}")

Chargement de l'architecture TrOCR...


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Chargement de la métrique d'évaluation (CER)...
Configuration du moteur d'entraînement...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0}.



🚀 DÉMARRAGE DE L'ENTRAÎNEMENT !
Une barre de progression va apparaître. Surveille la colonne 'cer' !


... (7 outputs left out)

There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].



💾 Sauvegarde du modèle final sur Google Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Entraînement terminé ! Modèle sauvegardé dans : /content/drive/MyDrive/OCR_Prescription/modele_trocr_medical_final


In [ ]:
# --- CELLULE 4 : L'ÉVALUATION FINALE SUR LE DATASET DE TEST ---

print("Préparation du dataset de Test (L'examen final)...")

# 1. On charge le dataset de test avec la même classe qu'avant
test_dataset = PrescriptionDataset(
    csv_file=os.path.join(ROOT_DIR, "test_trocr.csv"),
    image_folder=os.path.join(ROOT_DIR, "test/images"),
    processor=processor
)

print(f"Dataset chargé avec succès : {len(test_dataset)} images à lire.")
print("Lancement de l'évaluation finale. Le modèle lit les ordonnances inédites...")

# 2. On demande au Trainer d'évaluer ce nouveau dataset
resultats_test = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test"
)

# 3. Affichage du grand verdict !
print("\n" + "="*50)
print("🏆 RÉSULTATS OFFICIELS DE TON MODÈLE :")
print("="*50)
print(f"Score CER (Taux d'erreur sur les caractères) : {resultats_test['test_cer']:.4f}")
# On le traduit en pourcentage pour que ce soit plus parlant
precision = (1 - resultats_test['test_cer']) * 100
print(f"Précision globale de lecture : {precision:.2f} %")
print("="*50)

Préparation du dataset de Test (L'examen final)...
Dataset chargé avec succès : 780 images à lire.
Lancement de l'évaluation finale. Le modèle lit les ordonnances inédites...



🏆 RÉSULTATS OFFICIELS DE TON MODÈLE :
Score CER (Taux d'erreur sur les caractères) : 0.0310
Précision globale de lecture : 96.90 %
